# MST-GNN Paper Replication (Google Colab Notebook)

This notebook helps you run the replication code for **MST-GNN** (IEEE TCSS 2024) on **Google Colab (T4 GPU recommended)**.

### **Why run on Google Colab instead of your Mac Pro M1?**

1. **PyTorch Geometric (PyG) Compatibility**: PyG relies on highly-optimized custom sparse matrix operations (`torch-scatter`, `torch-sparse`). The PyTorch MPS (Metal Performance Shaders) backend on Apple Silicon M1 lacks stable or fully-optimized support for these sparse operations, which can lead to compilation errors or silent fallbacks to slow CPU processing.
2. **Hardware Acceleration**: NVIDIA CUDA (available on Colab T4) is natively supported by PyG with pre-compiled wheels. Training our dynamic graph models (where the graph structure changes daily) runs significantly faster on CUDA GPUs.
3. **Data Fetching**: AKShare fetching can be rate-limited, but running on a cloud instance sometimes behaves differently. However, once fetched, the datasets are cached as pickle files so future runs are instant.

## 1. Set Up Project Workspace

Choose **either** Option A (Google Drive) or Option B (GitHub) to load your files in Colab.

### **Option A: Google Drive**
Upload the `mst-gnn-impl` folder to your Google Drive, mount it, and navigate to it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# TODO: Adjust the path below to match where you uploaded the folder
%cd /content/drive/MyDrive/research/lab/mst-gnn-impl

### **Option B: Clone from GitHub**
Run the cell below to clone your public repository and navigate into the folder.

In [ ]:
!git clone https://github.com/quocnguyen5/mst-gnn-impl.git
%cd mst-gnn-impl

## 2. Install Dependencies

This cell dynamically installs PyTorch Geometric (PyG) and other packages matching the specific PyTorch and CUDA versions pre-installed in your Colab environment.

In [ ]:
# Install core python packages
!pip install akshare gensim jieba pandas numpy scikit-learn scipy matplotlib seaborn tqdm pyyaml tensorboard

# Install PyTorch Geometric matching the pre-installed PyTorch & CUDA version
import torch
pyt_version = torch.__version__.split('+')[0]
if torch.cuda.is_available():
    cuda_version = torch.version.cuda.replace('.', '')
    print(f"CUDA is available. Version: {torch.version.cuda}")
    !pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{pyt_version}+cu{cuda_version}.html
else:
    print("Warning: CUDA is NOT available. Running on CPU mode.")
    !pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{pyt_version}+cpu.html

!pip install torch-geometric

## 3. Run Experiments

### **A. Run the Main Experiment (CSI 300)**
This script handles:
1. Data collection via AKShare (and caches to raw folders).
2. Technical feature engineering (13 features).
3. Construction of 4 stock network layers (Shareholding, Industry, Topicality/News LDA, Comovement).
4. Splitting chronological snapshots into Train, Val, and Test.
5. Model training (using early stopping).
6. Backtesting and trading simulation.

In [ ]:
# Run CSI 300 with mean aggregator (fastest, standard variant)
!python -m experiments.run_main --dataset csi300 --aggregator mean

### **B. Run Ablation Studies**
This compares the full MST-GNN model with several ablation variants:
- Without STNA (no spatial-temporal neighborhood aggregation)
- Without HOFF (replaced with simple concatenation)
- Single-task configurations (movement direction only or ranking only)

In [ ]:
!python -m experiments.run_ablation --dataset csi300

### **C. Run Network Combination Analysis**
This evaluates the model when using different combinations of the 4 networks (Shareholding, Industry, Topicality, Comovement) to prove that multi-network aggregation improves results.

In [ ]:
!python -m experiments.run_network_analysis --dataset csi300

## 4. Visualize Results

This script plots and displays all saved charts (cumulative return backtest lines, ablation charts, and network combination bar plots).

In [ ]:
from IPython.display import Image, display
import os

results_dir = 'results'
if os.path.exists(results_dir):
    print("Generated figures in 'results/':")
    for file_name in sorted(os.listdir(results_dir)):
        if file_name.endswith('.png'):
            print(f"\nFigure: {file_name}")
            display(Image(filename=os.path.join(results_dir, file_name)))
else:
    print("No results directory found. Please run the experiments first.")